In [1]:
# 1. Datetimeindexcreation - 3 pathways

#there are 3 ways to create datetimeindex: strings, timestmps, ranges

# 2. .dt can help with extracting year, months, days etc. But note this only works on Series.

# 3. Partial String Indexing

# 4. Timezone-Naive vs Timezone-Aware

# 5. Business Day Calendars - real trading days
 
# 6. asfreq() vs .resample() 

# Notes on Auto Adjust Parameter on yf.download()

Excellent instinct to ask! This is **foundational** for calculating returns correctly. Let me break it down:

---

## **What Are Price Adjustments?**

Stock prices need to be adjusted for **corporate actions** that change the share structure:

### **1. Stock Splits**
- **Example:** Apple does a 4-for-1 split on Aug 31, 2020
- **Before split:** 1 share at $400
- **After split:** 4 shares at $100 each
- **Your wealth:** Unchanged (1 × $400 = 4 × $100)

**The problem:** If you're calculating returns and see the price drop from $400 to $100 overnight, naive code thinks you lost 75%. You didn't — the share count quadrupled.

### **2. Dividends**
- **Example:** Microsoft pays a $0.68 dividend per share
- **Ex-dividend date:** Stock price drops by ~$0.68 at open
- **Your wealth:** Unchanged (you got cash, stock dropped)

**The problem:** If you ignore the dividend, you see a -$0.68 price drop and think the stock underperformed. It didn't — you got paid.

---

## **Raw Prices vs Adjusted Prices**

**Raw prices** (what Bloomberg/exchange reports):
- What you'd actually pay if you bought the stock that day
- Reflects the **actual market price** at that moment
- **Not comparable across time** if there were splits/dividends

**Adjusted prices** (backwards-adjusted for splits/dividends):
- The entire price history is **retroactively adjusted** to account for corporate actions
- Makes historical returns **calculation-safe**
- Standard in quantitative analysis

**Example:**

```
Date         Raw Close    Adj Close    Event
2020-08-28   $499.23      $124.81      (before split)
2020-08-31   $129.04      $129.04      4:1 split happens
2020-09-01   $134.18      $134.18      (after split)
```

Notice: The Aug 28 **raw close** ($499.23) gets divided by 4 in the **Adj Close** column ($124.81) so you can calculate continuous returns across the split.

---

## **What Does `auto_adjust=True` Do?**

When you call `yf.download(..., auto_adjust=True)`:

**Before (default behavior):**
```python
data.columns = MultiIndex(['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'])
# You have BOTH raw 'Close' and 'Adj Close' columns
```

**After (`auto_adjust=True`):**
```python
data.columns = Index(['Open', 'High', 'Low', 'Close', 'Volume'])
# The 'Close' column IS the adjusted close
# No separate 'Adj Close' column — it's baked in
```

**Translation:** yfinance replaces the raw OHLC prices with **adjusted versions**, and drops the redundant `Adj Close` column.

---

## **Why This Matters for Returns**

Your return calculation depends on adjusted prices:

**Wrong (using raw prices):**
```python
# Across a 4:1 split, this shows -75% return (nonsense)
returns = data['Close'].pct_change()  # if Close is raw
```

**Right (using adjusted prices):**
```python
# Correctly shows actual investor return (continuous across split)
returns = data['Adj Close'].pct_change()  # or auto_adjust=True + 'Close'
```

**Your Module 3 Phase 1 work on returns:** Remember calculating daily returns with `.pct_change()`? If you used raw prices, every dividend/split would look like a crash or moon shot. Adjusted prices fix that.

---

## **The Trade-Off**

**Pros of `auto_adjust=True`:**
- Cleaner column structure (no MultiIndex)
- Default `'Close'` is calculation-ready
- Less cognitive load ("which column do I use?")

**Cons of `auto_adjust=True`:**
- You lose access to **raw historical prices** (can't see what the stock *actually* traded at)
- If you need raw OHLC for certain analyses (e.g., intraday volatility), you're stuck

**Best practice for quant work:** Use `auto_adjust=True` for **returns/backtesting**, keep raw data if you're doing **price-level analysis** (e.g., support/resistance levels in technical analysis).

---

## **Real-World Example**

Let's say you're backtesting a strategy from 2019-2024:

```python
# WITHOUT auto_adjust
data = yf.download('AAPL', start='2019-01-01', end='2024-12-31')
returns = data['Adj Close'].pct_change()  # Correct returns

# WITH auto_adjust (cleaner)
data = yf.download('AAPL', start='2019-01-01', end='2024-12-31', auto_adjust=True)
returns = data['Close'].pct_change()  # Same result, cleaner code
```

Both produce **identical returns** — the second is just less fiddly.

---

## **Quick Check Question for You**

**Scenario:** Tesla did a 3-for-1 split on Aug 25, 2022. The raw close on Aug 24 was $891.30.

1. What should the **Adj Close** for Aug 24 show after the split? (Hint: divide by split ratio)
2. If you calculate returns using **raw Close**, what spurious return do you see on Aug 25?

Take 30 seconds to think through it — this will lock in the concept.

---

**TL;DR:** `auto_adjust=True` bakes split/dividend adjustments into the OHLC prices, making `'Close'` safe for returns calculations. It's the right default for quant work unless you specifically need raw historical prices.

Does that click? Want me to show you how to verify the adjustments in your Apple data (they had a 4:1 split in 2020)?